# Notebook 1 — Data Profiling
### Project: Retail Sales & Customer Purchase Prediction (Group A)
### BCA 6th Semester Internship Project — Team of 5

**Purpose of this notebook:** Before any cleaning or modeling, we inspect the raw dataset exactly as it was
received. Nothing is modified here. This notebook produces the data-quality report that guides every
decision made later in `02_cleaning_and_eda.ipynb`.

We do **not** assume the data is clean. Every claim below is backed by code output run on the actual file.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

RAW_PATH = "../data/raw/online_retail_II.xlsx"
print("Reading raw file:", RAW_PATH)

Reading raw file: ../data/raw/online_retail_II.xlsx


## 1. Sheet Names

In [2]:
xls = pd.ExcelFile(RAW_PATH)
print("Sheets found in workbook:")
for s in xls.sheet_names:
    print(" -", s)

Sheets found in workbook:
 - Year 2009-2010
 - Year 2010-2011


## 2. Load Each Sheet Separately (raw, untouched)

In [3]:
df_2009_2010 = pd.read_excel(RAW_PATH, sheet_name="Year 2009-2010")
df_2010_2011 = pd.read_excel(RAW_PATH, sheet_name="Year 2010-2011")

print("Year 2009-2010 shape:", df_2009_2010.shape)
print("Year 2010-2011 shape:", df_2010_2011.shape)

Year 2009-2010 shape: (525461, 8)
Year 2010-2011 shape: (541910, 8)


## 3. Column Names — Are the Two Sheets the Same Schema?

In [4]:
print("Columns in 2009-2010:", list(df_2009_2010.columns))
print("Columns in 2010-2011:", list(df_2010_2011.columns))
print("\nSame column names and order:", list(df_2009_2010.columns) == list(df_2010_2011.columns))
print("Same dtypes:")
print(pd.DataFrame({"2009-2010": df_2009_2010.dtypes, "2010-2011": df_2010_2011.dtypes}))

Columns in 2009-2010: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
Columns in 2010-2011: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Same column names and order: True
Same dtypes:
                  2009-2010       2010-2011
Invoice              object          object
StockCode            object          object
Description          object          object
Quantity              int64           int64
InvoiceDate  datetime64[us]  datetime64[us]
Price               float64         float64
Customer ID         float64         float64
Country                 str             str


**Finding:** Both sheets have identical columns, order, and data types. They are simply two calendar
years of the same operational export (Dec 2009 -> Dec 2010, and Dec 2010 -> Dec 2011), so it is safe to
combine them after we finish profiling each one individually.

## 4. First 5 Rows of Each Sheet

In [5]:
print("Year 2009-2010:")
display(df_2009_2010.head())

Year 2009-2010:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [6]:
print("Year 2010-2011:")
display(df_2010_2011.head())

Year 2010-2011:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 5. Data Types

In [7]:
df_2009_2010.info()

<class 'pandas.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[us]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 32.1+ MB


In [8]:
df_2010_2011.info()

<class 'pandas.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      541910 non-null  object        
 1   StockCode    541910 non-null  object        
 2   Description  540456 non-null  object        
 3   Quantity     541910 non-null  int64         
 4   InvoiceDate  541910 non-null  datetime64[us]
 5   Price        541910 non-null  float64       
 6   Customer ID  406830 non-null  float64       
 7   Country      541910 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


## 6. Missing Values (per sheet)

In [9]:
print("Missing values - Year 2009-2010")
print(df_2009_2010.isnull().sum())
print("\nMissing values - Year 2010-2011")
print(df_2010_2011.isnull().sum())

Missing values - Year 2009-2010


Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

Missing values - Year 2010-2011


Invoice             0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
Price               0
Customer ID    135080
Country             0
dtype: int64


**Finding:** `Description` and `Customer ID` are the only columns with missing values in either sheet.
`Customer ID` is missing for roughly 20% of rows in both years — these are transactions we cannot attach
to a specific customer, which matters a great deal for a *customer* purchase-prediction project.

## 7. Duplicate Rows (per sheet, exact full-row duplicates)

In [10]:
print("Duplicate rows - Year 2009-2010:", df_2009_2010.duplicated().sum())
print("Duplicate rows - Year 2010-2011:", df_2010_2011.duplicated().sum())

Duplicate rows - Year 2009-2010: 6865


Duplicate rows - Year 2010-2011: 5268


## 8. Unique Values Check

In [11]:
for name, df in [("2009-2010", df_2009_2010), ("2010-2011", df_2010_2011)]:
    print(f"--- {name} ---")
    print("Unique Invoices:", df['Invoice'].nunique())
    print("Unique Customers:", df['Customer ID'].nunique())
    print("Unique Products (StockCode):", df['StockCode'].nunique())
    print("Unique Countries:", df['Country'].nunique())
    print()

--- 2009-2010 ---
Unique Invoices: 28816
Unique Customers: 4383
Unique Products (StockCode): 4632
Unique Countries: 40

--- 2010-2011 ---
Unique Invoices: 25900
Unique Customers: 4372
Unique Products (StockCode): 4070
Unique Countries: 38



## 9. Invalid / Negative Quantities

In [12]:
for name, df in [("2009-2010", df_2009_2010), ("2010-2011", df_2010_2011)]:
    print(f"--- {name} ---")
    print("Quantity < 0:", (df['Quantity'] < 0).sum())
    print("Quantity == 0:", (df['Quantity'] == 0).sum())
    print(df['Quantity'].describe())
    print()

--- 2009-2010 ---
Quantity < 0: 12326
Quantity == 0: 0
count    525461.000000
mean         10.337667
std         107.424110
min       -9600.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       19152.000000
Name: Quantity, dtype: float64

--- 2010-2011 ---
Quantity < 0: 10624
Quantity == 0: 0
count    541910.000000
mean          9.552234
std         218.080957
min      -80995.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64



## 10. Cancelled Invoices

In [13]:
for name, df in [("2009-2010", df_2009_2010), ("2010-2011", df_2010_2011)]:
    is_cancel = df['Invoice'].astype(str).str.startswith('C')
    print(f"--- {name} ---")
    print("Invoices flagged as cancellations (Invoice starts with 'C'):", is_cancel.sum())
    print("As % of rows:", round(100 * is_cancel.mean(), 2), "%")
    print()

--- 2009-2010 ---
Invoices flagged as cancellations (Invoice starts with 'C'): 10206
As % of rows: 1.94 %

--- 2010-2011 ---
Invoices flagged as cancellations (Invoice starts with 'C'): 9288
As % of rows: 1.71 %



**Finding:** Every cancelled row's Invoice number begins with the letter **C**, and cancelled rows carry
a **negative Quantity** (almost always). This is a clean, reliable signal we can use as a business rule in
the cleaning notebook: `is_cancelled = Invoice starts with 'C'`.

## 11. Zero / Negative Prices

In [14]:
for name, df in [("2009-2010", df_2009_2010), ("2010-2011", df_2010_2011)]:
    print(f"--- {name} ---")
    print("Price < 0:", (df['Price'] < 0).sum())
    print("Price == 0:", (df['Price'] == 0).sum())
    print()

--- 2009-2010 ---
Price < 0: 3


Price == 0: 3687

--- 2010-2011 ---
Price < 0: 2
Price == 0: 2515



**Finding:** A handful of rows have negative prices — these all belong to invoices starting with 'A'
(adjustment / bad-debt entries, e.g. `Invoice = A506401`, `StockCode = B` = "Adjust bad debt"), not real
product sales. Price == 0 rows are typically free items, samples, or manual adjustments and will need a
clear rule in cleaning.

## 12. Date Range

In [15]:
for name, df in [("2009-2010", df_2009_2010), ("2010-2011", df_2010_2011)]:
    print(f"--- {name} ---")
    print("Min date:", df['InvoiceDate'].min())
    print("Max date:", df['InvoiceDate'].max())
    print()

--- 2009-2010 ---
Min date: 2009-12-01 07:45:00
Max date: 2010-12-09 20:01:00

--- 2010-2011 ---
Min date: 2010-12-01 08:26:00
Max date: 2011-12-09 12:50:00



## 13. Countries

In [16]:
for name, df in [("2009-2010", df_2009_2010), ("2010-2011", df_2010_2011)]:
    print(f"--- {name} : {df['Country'].nunique()} countries ---")
    print(df['Country'].value_counts().head(10))
    print()

--- 2009-2010 : 40 countries ---
Country
United Kingdom     485852
EIRE                 9670
Germany              8129
France               5772
Netherlands          2769
Spain                1278
Switzerland          1187
Portugal             1101
Belgium              1054
Channel Islands       906
Name: count, dtype: int64

--- 2010-2011 : 38 countries ---
Country
United Kingdom    495478
Germany             9495
France              8558
EIRE                8196
Spain               2533
Netherlands         2371
Belgium             2069
Switzerland         2002
Portugal            1519
Australia           1259
Name: count, dtype: int64



## 14. Combine the Two Sheets (schema verified identical in Step 3)

In [17]:
df_raw_combined = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)
print("Combined shape:", df_raw_combined.shape)
print("Combined date range:", df_raw_combined['InvoiceDate'].min(), "to", df_raw_combined['InvoiceDate'].max())
print("Combined duplicate rows:", df_raw_combined.duplicated().sum())
print("Combined unique customers:", df_raw_combined['Customer ID'].nunique())
print("Combined unique invoices:", df_raw_combined['Invoice'].nunique())
print("Combined unique products:", df_raw_combined['StockCode'].nunique())

Combined shape: (1067371, 8)
Combined date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


Combined duplicate rows: 34335
Combined unique customers: 5942
Combined unique invoices: 53628
Combined unique products: 5305


## 15. Non-Product Stock Codes (administrative rows)

In [18]:
special_codes = ['POST', 'DOT', 'M', 'm', 'C2', 'D', 'BANK CHARGES', 'AMAZONFEE', 'PADS', 'CRUK']
mask_special = df_raw_combined['StockCode'].astype(str).isin(special_codes)
print("Administrative / non-product rows found:", mask_special.sum())
print(df_raw_combined.loc[mask_special, 'StockCode'].value_counts())

Administrative / non-product rows found: 5633
StockCode
POST            2122
DOT             1446
M               1421
C2               282
D                177
BANK CHARGES     102
AMAZONFEE         43
PADS              19
CRUK              16
m                  5
Name: count, dtype: int64


**Finding:** Codes such as `POST` (postage), `DOT`, `M`/`m` (manual entry), `BANK CHARGES`, `AMAZONFEE`,
`PADS`, `CRUK`, and `C2`/`D` are **not real products** — they are administrative or fee line items. These
need a documented decision in the cleaning notebook (most retail-analytics treatments exclude them from
product-level and customer-value analysis, since they distort "top products" and revenue-per-item figures).

## 16. Descriptive Statistics

In [19]:
df_raw_combined[['Quantity', 'Price']].describe()

,Quantity,Price
count,1.067371e+06,1.067371e+06
mean,9.938898e+00,4.649388e+00
std,1.727058e+02,1.235531e+02
min,-8.099500e+04,-5.359436e+04
25%,1.000000e+00,1.250000e+00
50%,3.000000e+00,2.100000e+00
75%,1.000000e+01,4.150000e+00
max,8.099500e+04,3.897000e+04


## 17. Initial Data-Quality Observations (Summary)

| Issue | Count (combined, 1,067,371 rows) | Decision needed in cleaning |
|---|---|---|
| Missing `Customer ID` | 243,007 (~22.8%) | Exclude from customer-level modeling; keep for product/revenue-level EDA only |
| Missing `Description` | 4,382 | Drop or leave blank depending on use |
| Full-row duplicates | 34,335 | Remove |
| Cancelled invoices (`Invoice` starts with `C`) | 19,494 | Exclude from "completed sales" revenue and modeling; can be analyzed separately |
| Adjustment invoices (`Invoice` starts with `A`) | 6 | Exclude — not real sales |
| Quantity < 0 (outside cancellations) | 3,457 | Investigate individually (e.g. `Description` = "lost", "damages", "?") — exclude from completed-sales set |
| Price <= 0 | 6,207 (6,202 zero + 5 negative) | Exclude from revenue calculations (free items / adjustments, not genuine sales) |
| Non-product `StockCode`s (POST, DOT, M, BANK CHARGES, etc.) | 5,633 | Exclude from product-level and revenue analysis |
| Two sheets share identical schema | — | Safe to combine after cleaning |
| Date coverage | 2009-12-01 to 2011-12-09 (~24 months, no gaps) | Supports a historical-period vs. target-period split for leakage-free modeling |

This data is **not clean** — it is a raw operational export with the kinds of issues any real transactional
dataset has. Every issue above will be handled explicitly and documented in `02_cleaning_and_eda.ipynb`,
never silently dropped.
